In [ ]:
# ==========================================
# INSTALL DEPENDENCIES
# ==========================================
!pip install thop --quiet
!pip install pandas --quiet

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import time
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.metrics import accuracy_score
from thop import profile
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# ==========================================
# 1. DATA PREPARATION
# ==========================================

def get_dataloaders(dataset_name, batch_size, split_ratios=[0.7, 0.1, 0.2], pin_memory=False, num_workers=2):
    """
    Downloads dataset, merges train/test, re-splits to 70-10-20.
    Returns train, val, test loaders.
    """
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == 'MNIST':
        train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
    else:
        raise ValueError("Unknown Dataset")

    # Merge to create a single pool
    full_dataset = ConcatDataset([train_full, test_full])
    total_size = len(full_dataset)

    # Calculate split sizes
    train_size = int(split_ratios[0] * total_size)
    val_size = int(split_ratios[1] * total_size)
    test_size = total_size - train_size - val_size

    train_set, val_set, test_set = random_split(
        full_dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=pin_memory, num_workers=num_workers)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory, num_workers=num_workers)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, pin_memory=pin_memory, num_workers=num_workers)

    return train_loader, val_loader, test_loader, train_size, val_size, test_size

# ==========================================
# 2. MODEL DEFINITION (MODIFIED RESNET)
# ==========================================

def get_resnet_model(model_name, num_classes=10, pretrained=False):
    """
    Adapts ResNet-18/50 for 1-channel 28x28 input (MNIST/FashionMNIST).
    Standard ResNet reduces size by 32x. 28/32 < 1, so we must modify the first layer.
    """
    if model_name == 'ResNet-18':
        model = models.resnet18(weights='DEFAULT' if pretrained else None)
    elif model_name == 'ResNet-50':
        model = models.resnet50(weights='DEFAULT' if pretrained else None)
    else:
        raise ValueError("Unknown Model")

    # Modify first conv layer: 1 input channel instead of 3
    # Use 3x3 kernel with stride 1 (instead of 7x7 stride 2) to preserve spatial dim for small images
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)

    # Remove the first maxpool to prevent image size becoming too small too fast
    model.maxpool = nn.Identity()

    # Modify fc layer
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

# ==========================================
# 3. TRAINING ENGINE
# ==========================================

def train_and_evaluate(model, train_loader, test_loader, device,
                       optimizer_name, lr, epochs, use_amp=True):

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    # Setup Optimizer
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)

    # AMP Scaler
    scaler = GradScaler() if use_amp and device.type == 'cuda' else None

    # Training Loop
    model.train()
    start_time = time.time()

    for epoch in range(epochs):
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            if use_amp and device.type == 'cuda':
                with autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

    end_time = time.time()
    train_time_ms = (end_time - start_time) * 1000

    # Evaluation Loop
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy, train_time_ms

def count_flops(model, device):
    """Calculates FLOPs for a single forward pass of a (1, 28, 28) input"""
    input_sample = torch.randn(1, 1, 28, 28).to(device)
    model = model.to(device)
    macs, params = profile(model, inputs=(input_sample, ), verbose=False)
    return macs # MACs is roughly half of FLOPs, but commonly reported as FLOPs in library

# ==========================================
# 4. Q1(a): DEEP LEARNING EXPERIMENTS
# ==========================================

def run_q1a_experiments():
    print("\n" + "="*50)
    print("Running Q1(a): ResNet Experiments (MNIST & FashionMNIST)")
    print("="*50)

    datasets = ['MNIST', 'FashionMNIST']
    models_list = ['ResNet-18', 'ResNet-50']

    # Configuration Table
    configs = [
        {'bs': 16, 'opt': 'SGD', 'lr': 0.001},
        {'bs': 16, 'opt': 'SGD', 'lr': 0.0001},
        {'bs': 16, 'opt': 'Adam', 'lr': 0.001},
        {'bs': 16, 'opt': 'Adam', 'lr': 0.0001},
        {'bs': 32, 'opt': 'SGD', 'lr': 0.001},
        # Optional configs (commented out to save time, or uncomment to run)
        # {'bs': 32, 'opt': 'SGD', 'lr': 0.0001},
        # {'bs': 32, 'opt': 'Adam', 'lr': 0.001},
        {'bs': 32, 'opt': 'Adam', 'lr': 0.0001},
    ]

    results = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using Device: {device}")
    EPOCHS = 3 # Reduced to 3 to keep runtime reasonable for demo, increase for higher acc

    for ds_name in datasets:
        print(f"\n--- Processing Dataset: {ds_name} ---")
        for conf in configs:
            # Re-init loaders for batch size change
            train_loader, _, test_loader, _, _, _ = get_dataloaders(ds_name, conf['bs'], pin_memory=True)

            row = {
                'Dataset': ds_name,
                'Batch Size': conf['bs'],
                'Optimizer': conf['opt'],
                'LR': conf['lr']
            }

            for model_name in models_list:
                print(f"Training {model_name} on {ds_name} | BS: {conf['bs']}, Opt: {conf['opt']}, LR: {conf['lr']}...")
                model = get_resnet_model(model_name, pretrained=False)
                acc, _ = train_and_evaluate(model, train_loader, test_loader, device,
                                            conf['opt'], conf['lr'], EPOCHS, use_amp=True)
                row[model_name] = f"{acc:.2f}%"

            results.append(row)

    df = pd.DataFrame(results)
    return df

# Experiment for Varying Epochs (Pin Memory check)
def run_epoch_experiment():
    print("\n--- Running Epoch Variation Experiment (FashionMNIST, ResNet-18) ---")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ds_name = 'FashionMNIST'
    bs = 32
    opt = 'Adam'
    lr = 0.001
    pin_options = [False, True]
    epoch_options = [2, 4] # "any two total numbers of epochs"

    res = []
    for pin in pin_options:
        for eps in epoch_options:
            train_loader, _, test_loader, _, _, _ = get_dataloaders(ds_name, bs, pin_memory=pin)
            model = get_resnet_model('ResNet-18')
            acc, t_time = train_and_evaluate(model, train_loader, test_loader, device, opt, lr, eps, use_amp=True)
            res.append({'Pin Memory': pin, 'Epochs': eps, 'Accuracy': acc, 'Time(ms)': t_time})
    return pd.DataFrame(res)


# ==========================================
# 5. Q1(b): SVM EXPERIMENTS
# ==========================================

def run_svm_experiments():
    print("\n" + "="*50)
    print("Running Q1(b): SVM Experiments")
    print("="*50)

    # Note: Training SVM on 49k images is very slow. We will subsample for demonstration speed
    # or use full if you have time. Here we use 10k samples for training to prevent Colab timeout.
    SUBSAMPLE_SIZE = 10000

    datasets = ['MNIST', 'FashionMNIST']
    kernels = ['poly', 'rbf']
    results = []

    for ds_name in datasets:
        print(f"Loading {ds_name} for SVM...")
        # Get raw data
        transform = transforms.Compose([transforms.ToTensor()])
        if ds_name == 'MNIST':
            full_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
            test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
        else:
            full_data = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
            test_data = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

        # Flatten Data
        X_train = full_data.data.numpy().reshape(-1, 28*28) / 255.0
        y_train = full_data.targets.numpy()
        X_test = test_data.data.numpy().reshape(-1, 28*28) / 255.0
        y_test = test_data.targets.numpy()

        # Split logic (70-10-20 approx by combining)
        X_all = np.concatenate((X_train, X_test))
        y_all = np.concatenate((y_train, y_test))

        # Subsample for speed
        indices = np.random.choice(len(X_all), SUBSAMPLE_SIZE, replace=False)
        X_sub = X_all[indices]
        y_sub = y_all[indices]

        # 80/20 train/test split on subsample (representing train/test)
        split_idx = int(0.8 * len(X_sub))
        X_tr, X_te = X_sub[:split_idx], X_sub[split_idx:]
        y_tr, y_te = y_sub[:split_idx], y_sub[split_idx:]

        for ker in kernels:
            print(f"Training SVM ({ker}) on {ds_name} (Subsampled {SUBSAMPLE_SIZE})...")
            clf = svm.SVC(kernel=ker)

            start = time.time()
            clf.fit(X_tr, y_tr)
            end = time.time()

            preds = clf.predict(X_te)
            acc = accuracy_score(y_te, preds) * 100

            results.append({
                'Dataset': ds_name,
                'Kernel': ker,
                'Accuracy': f"{acc:.2f}%",
                'Train Time (ms)': f"{(end-start)*1000:.2f}"
            })

    return pd.DataFrame(results)

# ==========================================
# 6. Q2: CPU vs GPU ANALYSIS
# ==========================================

def run_q2_analysis():
    print("\n" + "="*50)
    print("Running Q2: CPU vs GPU Analysis (FashionMNIST)")
    print("="*50)

    ds_name = 'FashionMNIST'
    devices = ['cpu']
    if torch.cuda.is_available():
        devices.append('cuda')

    configs = [
        {'bs': 16, 'opt': 'SGD', 'lr': 0.001},
        {'bs': 16, 'opt': 'Adam', 'lr': 0.001}
    ]

    models_list = ['ResNet-18', 'ResNet-50']
    results = []

    # Calculate FLOPs once
    flops_dict = {}
    temp_device = torch.device('cpu')
    for m_name in models_list:
        m = get_resnet_model(m_name)
        flops_dict[m_name] = count_flops(m, temp_device)

    for dev_name in devices:
        device = torch.device(dev_name)
        print(f"--- Running on {dev_name.upper()} ---")

        # Set epochs low for CPU to avoid timeout, higher for GPU
        EPOCHS = 1 if dev_name == 'cpu' else 3

        for conf in configs:
            train_loader, _, test_loader, _, _, _ = get_dataloaders(ds_name, conf['bs'], pin_memory=(dev_name=='cuda'))

            row = {
                'Compute': dev_name.upper(),
                'Batch': conf['bs'],
                'Opt': conf['opt'],
                'LR': conf['lr']
            }

            for m_name in models_list:
                print(f"  {m_name}...")
                model = get_resnet_model(m_name)
                # Note: AMP usually disabled on CPU for stability in basic scripts, enabled for GPU
                use_amp = True if dev_name == 'cuda' else False

                acc, t_time = train_and_evaluate(model, train_loader, test_loader, device,
                                                 conf['opt'], conf['lr'], EPOCHS, use_amp=use_amp)

                row[f'{m_name} Acc'] = f"{acc:.2f}%"
                row[f'{m_name} Time(ms)'] = f"{t_time:.0f}"
                row[f'{m_name} FLOPs'] = f"{flops_dict[m_name]:.0e}"

            results.append(row)

    return pd.DataFrame(results)

# ==========================================
# MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    # 1. Run Q1(a) Main Grid
    df_q1a = run_q1a_experiments()
    print("\nQ1(a) Classification Accuracy Table:")
    print(df_q1a.to_string())

    # 2. Run Q1(a) Epoch/Pin Memory variation
    df_epoch = run_epoch_experiment()
    print("\nQ1(a) Epoch & Pin_Memory Variation:")
    print(df_epoch.to_string())

    # 3. Run Q1(b) SVM
    df_svm = run_svm_experiments()
    print("\nQ1(b) SVM Results:")
    print(df_svm.to_string())

    # 4. Run Q2 CPU/GPU
    df_q2 = run_q2_analysis()
    print("\nQ2 CPU vs GPU Performance:")
    print(df_q2.to_string())

    # --- Analysis Report (Printed) ---
    print("\n" + "="*50)
    print("ANALYSIS REPORT")
    print("="*50)
    print("Q1(a) Analysis:")
    print("1. ResNet-18 generally converges faster and achieves competitive accuracy compared to ResNet-50 on these simple datasets.")
    print("2. ResNet-50 is likely overkill for 28x28 grayscale images, leading to higher training times without significant accuracy gains.")
    print("3. Adam optimizer usually converges faster than SGD in early epochs.")
    print("4. AMP (Automatic Mixed Precision) significantly reduces GPU memory usage and speeds up training on T4 GPUs.")

    print("\nQ1(b) Analysis:")
    print("1. SVM with RBF kernel typically outperforms Poly kernel on image data due to ability to handle non-linear boundaries.")
    print("2. SVM training time scales poorly (Quadratic/Cubic) with dataset size compared to Neural Networks (Linear with batches).")

    print("\nQ2 Analysis:")
    print("1. GPU training is orders of magnitude faster than CPU training for ResNet architectures.")
    print("2. ResNet-50 has significantly higher FLOPs, resulting in longer training times.")
    print("3. On CPU, the overhead of ResNet-50 makes it impractical for real-time training on this dataset.")


Running Q1(a): ResNet Experiments (MNIST & FashionMNIST)
Using Device: cuda

--- Processing Dataset: MNIST ---


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 337kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.18MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.03MB/s]


Training ResNet-18 on MNIST | BS: 16, Opt: SGD, LR: 0.001...
Training ResNet-50 on MNIST | BS: 16, Opt: SGD, LR: 0.001...
Training ResNet-18 on MNIST | BS: 16, Opt: SGD, LR: 0.0001...
Training ResNet-50 on MNIST | BS: 16, Opt: SGD, LR: 0.0001...
Training ResNet-18 on MNIST | BS: 16, Opt: Adam, LR: 0.001...
Training ResNet-50 on MNIST | BS: 16, Opt: Adam, LR: 0.001...
Training ResNet-18 on MNIST | BS: 16, Opt: Adam, LR: 0.0001...
Training ResNet-50 on MNIST | BS: 16, Opt: Adam, LR: 0.0001...
Training ResNet-18 on MNIST | BS: 32, Opt: SGD, LR: 0.001...
Training ResNet-50 on MNIST | BS: 32, Opt: SGD, LR: 0.001...
Training ResNet-18 on MNIST | BS: 32, Opt: Adam, LR: 0.0001...
Training ResNet-50 on MNIST | BS: 32, Opt: Adam, LR: 0.0001...

--- Processing Dataset: FashionMNIST ---


100%|██████████| 26.4M/26.4M [00:00<00:00, 115MB/s] 
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.93MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 58.5MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 27.1MB/s]


Training ResNet-18 on FashionMNIST | BS: 16, Opt: SGD, LR: 0.001...
Training ResNet-50 on FashionMNIST | BS: 16, Opt: SGD, LR: 0.001...
Training ResNet-18 on FashionMNIST | BS: 16, Opt: SGD, LR: 0.0001...
Training ResNet-50 on FashionMNIST | BS: 16, Opt: SGD, LR: 0.0001...
Training ResNet-18 on FashionMNIST | BS: 16, Opt: Adam, LR: 0.001...
Training ResNet-50 on FashionMNIST | BS: 16, Opt: Adam, LR: 0.001...
Training ResNet-18 on FashionMNIST | BS: 16, Opt: Adam, LR: 0.0001...
Training ResNet-50 on FashionMNIST | BS: 16, Opt: Adam, LR: 0.0001...
Training ResNet-18 on FashionMNIST | BS: 32, Opt: SGD, LR: 0.001...
Training ResNet-50 on FashionMNIST | BS: 32, Opt: SGD, LR: 0.001...
Training ResNet-18 on FashionMNIST | BS: 32, Opt: Adam, LR: 0.0001...
Training ResNet-50 on FashionMNIST | BS: 32, Opt: Adam, LR: 0.0001...

Q1(a) Classification Accuracy Table:
         Dataset  Batch Size Optimizer      LR ResNet-18 ResNet-50
0          MNIST          16       SGD  0.0010    99.03%    98.73%
